## Model Training

We load the merged dataset, drop invalid rows, normalize the landmark coordinates, augment the training split, then train and tune six classifiers.

### Normalization

Extraction already centers each sample around the wrist (landmark 0 subtracted). We add one more step: divide every coordinate by the distance from wrist to middle finger MCP (landmark 9). This makes the features scale-invariant so hand size doesn't affect predictions.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/asl_project/data/landmarks_merged.csv')
df = df[df['letter'] != 'NOTHING'].copy()

feat_cols = [f'x{i}' for i in range(21)] + [f'y{i}' for i in range(21)] + [f'z{i}' for i in range(21)]
required_cols = feat_cols + ['letter']

rows_before_cleaning = len(df)
missing_rows = int(df[required_cols].isna().any(axis=1).sum())
df = df.dropna(subset=required_cols).copy()

duplicate_rows = int(df.duplicated(subset=required_cols).sum())
df = df.drop_duplicates(subset=required_cols).copy()

class_counts = df['letter'].value_counts()
print(f'rows after removing NOTHING: {rows_before_cleaning}')
print(f'missing rows removed: {missing_rows}')
print(f'duplicate rows removed: {duplicate_rows}')
print(f'rows after cleaning: {len(df)}')
print(f'classes: {df["letter"].nunique()}')
print(f'class size: min={class_counts.min()} max={class_counts.max()} ratio={class_counts.max() / class_counts.min():.2f}')

X = df[feat_cols].values.astype(np.float32)
y = df['letter'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def scale_normalize(X):
    # landmark 9: x at col 9, y at col 30 (21+9)
    ref_x = X[:, 9]
    ref_y = X[:, 30]
    scale = np.sqrt(ref_x**2 + ref_y**2)
    scale = np.where(scale < 1e-6, 1.0, scale)
    return X / scale[:, np.newaxis]

X_train = scale_normalize(X_train)
X_test  = scale_normalize(X_test)

print(f'train: {X_train.shape}  test: {X_test.shape}')

### Data Augmentation

We generate two extra copies of the training set with random perturbations. This helps the model handle slight variations in hand angle and camera distance without needing more raw data.

- **Rotation +-15 degrees** in the xy plane
- **Gaussian noise** (sigma=0.01)
- **Mirror** (flip x) with 50% probability

Each augmented copy gets re-normalized after perturbation so it stays consistent with what the app sends to the model at inference time. Augmentation is training-only — the test set is untouched.

In [ ]:
def augment(X, y, n_copies=2, seed=42):
    rng = np.random.default_rng(seed)
    aug_X, aug_y = [X], [y]

    for _ in range(n_copies):
        X_aug = X.copy()
        n = len(X_aug)

        # rotation in xy
        angles  = rng.uniform(-15, 15, n) * np.pi / 180
        cos_a, sin_a = np.cos(angles), np.sin(angles)
        xs = X_aug[:, :21].copy()
        ys = X_aug[:, 21:42].copy()
        X_aug[:, :21]   = cos_a[:, None] * xs - sin_a[:, None] * ys
        X_aug[:, 21:42] = sin_a[:, None] * xs + cos_a[:, None] * ys

        # noise
        X_aug += rng.normal(0, 0.01, X_aug.shape)

        # mirror x
        flip = rng.random(n) < 0.5
        X_aug[flip, :21] *= -1

        # keep augmented features consistent with deployment preprocessing
        X_aug = scale_normalize(X_aug)

        aug_X.append(X_aug)
        aug_y.append(y)

    return np.concatenate(aug_X), np.concatenate(aug_y)

X_train_aug, y_train_aug = augment(X_train, y_train, n_copies=2)
print(f'train after augmentation: {X_train_aug.shape}')

### Training Six Classifiers

We run logistic regression, KNN, random forest, SVM, MLP, and LightGBM. All six train on the augmented set and score on the held-out test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
import time

le = LabelEncoder()
y_tr = le.fit_transform(y_train)
y_tr_aug = le.transform(y_train_aug)
y_te = le.transform(y_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42),
    'MLP':                 MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300, random_state=42),
    'LightGBM':            LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
}

results = {}
for name, clf in models.items():
    t0 = time.time()
    clf.fit(X_train_aug, y_tr_aug)
    acc = clf.score(X_test, y_te)
    elapsed = time.time() - t0
    results[name] = {'acc': acc, 'clf': clf}
    print(f'{name:22s}  acc={acc:.4f}  ({elapsed:.1f}s)')

### Hyperparameter Tuning

CV runs on the original (non-augmented) training split to avoid a subtle leak: if we CV on the augmented set, copies of held-out samples end up in the training fold and inflate the scores. Best params get selected from clean CV, then the winner is refit on the full augmented training data.

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')

tuned = {}

print('tuning LR...')
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {'C': [0.1, 1, 10], 'solver': ['lbfgs', 'saga']},
    cv=3, n_jobs=-1
)
grid.fit(X_train, y_tr)
tuned['Logistic Regression'] = grid.best_estimator_
tuned['Logistic Regression'].fit(X_train_aug, y_tr_aug)
print(f'  {grid.best_params_}  cv={grid.best_score_:.4f}')

print('tuning KNN...')
grid = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': [3, 5, 7, 11], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']},
    cv=3, n_jobs=-1
)
grid.fit(X_train, y_tr)
tuned['KNN'] = grid.best_estimator_
tuned['KNN'].fit(X_train_aug, y_tr_aug)
print(f'  {grid.best_params_}  cv={grid.best_score_:.4f}')

print('tuning Random Forest...')
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {'n_estimators': [200, 300], 'max_depth': [None, 20, 40], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, 4]},
    n_iter=15, cv=3, random_state=42, n_jobs=-1
)
search.fit(X_train, y_tr)
tuned['Random Forest'] = search.best_estimator_
tuned['Random Forest'].fit(X_train_aug, y_tr_aug)
print(f'  {search.best_params_}  cv={search.best_score_:.4f}')

print('tuning SVM...')
grid = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42),
    {'C': [1, 10, 100], 'gamma': ['scale', 'auto']},
    cv=3, n_jobs=-1, verbose=1
)
grid.fit(X_train, y_tr)
tuned['SVM'] = grid.best_estimator_
tuned['SVM'].fit(X_train_aug, y_tr_aug)
print(f'  {grid.best_params_}  cv={grid.best_score_:.4f}')

print('tuning MLP...')
search = RandomizedSearchCV(
    MLPClassifier(max_iter=300, random_state=42, early_stopping=True),
    {
        'hidden_layer_sizes': [(256, 128), (256, 128, 64), (512, 256), (512, 256, 128)],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate_init': [0.001, 0.0005]
    },
    n_iter=12, cv=3, random_state=42
)
search.fit(X_train, y_tr)
tuned['MLP'] = search.best_estimator_
tuned['MLP'].fit(X_train_aug, y_tr_aug)
print(f'  {search.best_params_}  cv={search.best_score_:.4f}')

print('tuning LightGBM...')
grid = GridSearchCV(
    LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
    {'num_leaves': [31, 63, 127], 'learning_rate': [0.05, 0.1], 'min_child_samples': [10, 20]},
    cv=3, n_jobs=-1
)
grid.fit(X_train, y_tr)
tuned['LightGBM'] = grid.best_estimator_
tuned['LightGBM'].fit(X_train_aug, y_tr_aug)
print(f'  {grid.best_params_}  cv={grid.best_score_:.4f}')

### Final Comparison

In [ ]:
final_scores = {}

print(f'{"model":22s}  {"test acc":>10}')
print('-' * 36)
for name, clf in tuned.items():
    acc = clf.score(X_test, y_te)
    final_scores[name] = acc
    print(f'{name:22s}  {acc:.4f}')

best_model_name = max(final_scores, key=final_scores.get)
best_model = tuned[best_model_name]
print(f'\nbest: {best_model_name} ({final_scores[best_model_name]:.4f})')

### Evaluation — Confusion Matrix and Per-Class Report

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

y_pred = best_model.predict(X_test)

print(classification_report(y_te, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.3, ax=ax)
ax.set_xlabel('predicted')
ax.set_ylabel('actual')
ax.set_title(f'confusion matrix — tuned {best_model_name}')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/asl_project/confusion_matrix.png', dpi=150)
plt.show()

### Save Model

Downloads folder: grab `model.pkl` and `label_encoder.pkl` from Drive and drop them next to `app.py` to run the demo.

In [ ]:
import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print(f'saved {best_model_name} -> model.pkl')
print('download model.pkl and label_encoder.pkl from the Colab file browser and put them next to app.py')